# Random Forests

A random forest is an **ensemble** of many decision trees. Each tree is trained on a random subset of the rows *and* a random subset of the features, so no two trees are identical. Every tree makes its own prediction, and the forest returns the **majority vote** (classification) or the **average** (regression) - so the idiosyncratic mistakes of any one tree get averaged away.

**Topics covered in this notebook**

1. Intuition - bagging & an ensemble of *decorrelated* trees
2. Training on a dataset (wine cultivars)
3. Inspecting & interpreting the model (feature importance, out-of-bag score, confusion matrix)
4. When to use it

## 1. Intuition

**The problem with one tree.** A single deep decision tree can memorise its training data - it keeps splitting until each leaf is pure. That gives it *low bias* but *high variance*: reshuffle the training rows a little and you get a very different tree. It overfits.

**Bagging (Bootstrap AGGregatING).** Instead of one tree, grow many, each on a **bootstrap sample** - a dataset the same size as the original, drawn *with replacement* (so some rows repeat and about a third are left out). Averaging the predictions of many high-variance, roughly-unbiased models keeps the low bias but slashes the variance. Intuitively, if each tree's error is partly random, averaging many of them cancels the noise:

$$\text{Var}(\bar{T}) \;=\; \rho\,\sigma^2 \;+\; \frac{1-\rho}{B}\,\sigma^2$$

where $B$ is the number of trees, $\sigma^2$ is a single tree's variance, and $\rho$ is the *correlation* between trees. More trees ($B\uparrow$) shrinks the second term - but only if the trees are not perfectly correlated.

**The extra trick that makes it a *forest*.** At every split, a random forest considers only a **random subset of the features** (by default $\sqrt{n}$ of them for classification). This stops one or two strong features from dominating every tree, so the trees end up **decorrelated** ($\rho\downarrow$) - which drives that first variance term down too. Decorrelated trees are the whole point: they are what makes the forest beat a single tuned tree.

**Out-of-bag (OOB) estimate.** Because each tree leaves out ~1/3 of the rows (its "out-of-bag" samples), we can score each row using only the trees that never saw it. Averaging those gives a free, built-in validation score - no separate hold-out needed.

## 2. Training on a Dataset

We classify wines into **three cultivars** using 13 chemical measurements (alcohol, flavanoids, colour intensity, ...). The `n_estimators` argument sets how many trees to grow.

In [ ]:
# Core tools for this lesson:
from sklearn.datasets import load_wine            # small built-in dataset (no download needed)
from sklearn.model_selection import train_test_split  # split rows into train / test
from sklearn.ensemble import RandomForestClassifier    # the forest itself
from sklearn.metrics import accuracy_score        # fraction of correct predictions
import matplotlib.pyplot as plt                    # plotting (feature importances, confusion matrix)

In [ ]:
import pandas as pd   # only used to view the raw features as a tidy table

In [ ]:
# load_wine() returns a Bunch: .data (features), .target (labels 0/1/2),
# .feature_names (the 13 chemical measurements) and .target_names (the 3 cultivars).
wine = load_wine()

In [ ]:
# Wrap the raw feature matrix in a DataFrame just so we can eyeball it with column names.
# (The model trains on the NumPy arrays directly - the DataFrame is purely for display.)
df = pd.DataFrame(data=wine.data, columns=wine.feature_names)

In [ ]:
df.head()   # first 5 rows: one wine per row, one chemical measurement per column

In [ ]:
# X = feature matrix (178 wines x 13 features), y = the cultivar label (0, 1 or 2).
X, y = wine.data, wine.target

# Hold out 20% of the wines for testing so we can measure performance on data the
# forest never trained on. random_state fixes the shuffle so the split is reproducible.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Build the forest:
#   n_estimators=100 -> grow 100 decision trees (more trees = steadier votes, diminishing returns).
#   oob_score=True   -> also compute the out-of-bag score for free (see next cell).
#                       This works because bootstrap=True by default, so each tree leaves ~1/3 out.
#   random_state=42  -> makes the bootstrap sampling and per-split feature choices reproducible.
# Note: no feature scaling is needed - trees split on thresholds, so units/ranges don't matter.
forest = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)

# fit() grows all 100 trees: each on its own bootstrap sample, each split picking from a
# random subset of the 13 features.
forest.fit(X_train, y_train)

In [ ]:
# predict() runs every test wine through all 100 trees and returns the majority-vote class.
preds = forest.predict(X_test)

In [ ]:
# accuracy_score = fraction of test wines the forest labelled correctly.
print("test accuracy:", round(accuracy_score(y_test, preds), 3))

# The forest is literally a list of fitted trees; len() confirms we grew 100.
print("trees in the forest:", len(forest.estimators_))

# The out-of-bag score: each training wine scored only by the trees that never saw it.
# It's a built-in, no-extra-data estimate of generalisation - handy as a sanity check
# that lines up nicely with the held-out test accuracy above.
print("out-of-bag score:", round(forest.oob_score_, 3))

## 3. Inspecting & Interpreting the Model

### Feature importance

As it builds each tree, the forest tracks how much every split on a given feature reduces impurity (Gini). Averaging that reduction across *all* splits in *all* trees gives `feature_importances_` - a normalised ranking (it sums to 1) of how much the model leaned on each feature. It's a quick, reliable way to see which measurements actually drive the classification.

*Caveat:* impurity-based importance can be biased toward high-cardinality / continuous features, so treat it as a guide, not gospel.

In [ ]:
import numpy as np

# feature_importances_ is one number per feature (13 values that sum to 1.0).
importances = forest.feature_importances_

# argsort() sorts ascending and returns indices; [::-1] reverses to descending;
# [:5] keeps the 5 most important features.
order = np.argsort(importances)[::-1][:5]   # indices of the top-5 features, best first

In [ ]:
# Print the top-5 features with their importance scores, aligned into a neat column.
print("Top 5 features by importance:")
for i in order:
    print(f"  {wine.feature_names[i]:30s} {importances[i]:.3f}")

In [ ]:
# Plot ALL 13 importances as a horizontal bar chart, sorted so the strongest is on top.
full_order = np.argsort(importances)   # ascending: barh draws bottom-up, so smallest at bottom

fig, ax = plt.subplots(figsize=(7, 5))                 # this cell owns its own figure/axes
ax.barh(range(len(importances)), importances[full_order], color="#4c72b0")
ax.set_yticks(range(len(importances)))                 # one tick per feature...
ax.set_yticklabels([wine.feature_names[i] for i in full_order])  # ...labelled with its name
ax.set_xlabel("importance (impurity reduction, sums to 1)")
ax.set_title("Random forest feature importances")
fig.tight_layout()
plt.show()

### Checking the predictions

Beyond a single accuracy number, we inspect *where* the model is right or wrong. The **confusion matrix** cross-tabulates true vs. predicted cultivars (perfect = everything on the diagonal), and the **classification report** breaks precision / recall / F1 down per class.

In [ ]:
# Sanity peek: line up the first 5 predictions against the true labels.
print("predicted: ", preds[:5])
print("actual   : ", y_test[:5])

In [ ]:
from sklearn.metrics import confusion_matrix

# Row = true class, column = predicted class. Off-diagonal entries are mistakes.
cm = confusion_matrix(y_test, preds)

# Draw it as a heatmap so the diagonal jumps out (own figure/axes for this cell).
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")                       # colour intensity = count
ax.set_xticks(range(len(wine.target_names)))
ax.set_yticks(range(len(wine.target_names)))
ax.set_xticklabels(wine.target_names)                  # predicted cultivar names on x
ax.set_yticklabels(wine.target_names)                  # true cultivar names on y
ax.set_xlabel("predicted"); ax.set_ylabel("actual")
ax.set_title("Confusion matrix")
# Annotate each cell with its count; pick a readable text colour for dark vs light cells.
for r in range(cm.shape[0]):
    for c in range(cm.shape[1]):
        ax.text(c, r, cm[r, c], ha="center", va="center",
                color="white" if cm[r, c] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Per-class precision (of the wines I called class k, how many were?),
# recall (of the true class-k wines, how many did I catch?) and their F1 harmonic mean.
# target_names swaps the 0/1/2 row labels for the human-readable cultivar names.
c_report = classification_report(y_test, preds, target_names=wine.target_names)
print(c_report)

## 4. When to Use It

- A strong, low-effort **default** for most tabular classification / regression problems - often the first thing to reach for.
- Captures non-linearities and feature interactions with almost no tuning, and needs **no feature scaling** (trees split on thresholds).
- Far more **robust to overfitting** than a single tree, thanks to bagging + decorrelated trees averaging out variance.
- Gives you a free **out-of-bag** validation estimate and a built-in **feature-importance** ranking.

**Trade-offs**

- Less interpretable than one tree - you trade the readable flowchart for a committee of hundreds.
- Larger and slower to predict as `n_estimators` grows (memory + latency).
- On very high-dimensional sparse data (e.g. text) or when you need the last few % of accuracy, gradient-boosted trees often win.

**Key knobs:** `n_estimators` (more trees = steadier, diminishing returns), `max_depth` (cap tree size), `max_features` (how many features each split may consider - the decorrelation dial).